# Prediction Analysis — US Egg Farms (chicken_eggs_united_states)

Investigate false positives, compare prediction types, and visualize sample patches.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# Adjust if running locally vs on pod
DATA_ROOT = Path("/workspace/farm-mapping")
if not DATA_ROOT.exists():
    DATA_ROOT = Path("..") 

PATCHES_DIR = DATA_ROOT / "data" / "patches"
SCORED_PATH = PATCHES_DIR / "scored_candidates.parquet"
SPLIT_PATH = PATCHES_DIR / "split_assignments.csv"
PATCH_META_PATH = PATCHES_DIR / "patch_meta.csv"

print(f"Data root: {DATA_ROOT}")
print(f"Scored exists: {SCORED_PATH.exists()}")
print(f"Split exists: {SPLIT_PATH.exists()}")
print(f"Patch meta exists: {PATCH_META_PATH.exists()}")

In [ ]:
# Load scored candidates and classify
scored = gpd.read_parquet(SCORED_PATH)
patch_meta = pd.read_csv(PATCH_META_PATH)

def classify(row):
    pred, true = int(row.get("predicted_label", 0)), int(row.get("true_label", 0))
    if pred == 1 and true == 1: return "TP"
    if pred == 1 and true == 0: return "FP"
    if pred == 0 and true == 1: return "FN"
    return "TN"

scored["pred_class"] = scored.apply(classify, axis=1)

# Merge patch_path so we can load .npy files
scored = scored.merge(
    patch_meta[["candidate_id", "patch_path"]].drop_duplicates("candidate_id"),
    on="candidate_id", how="left"
)

print(f"Total: {len(scored)}")
print(scored["pred_class"].value_counts())
print(f"\nSplits: {scored['split'].value_counts().to_dict()}")

## 1. Score Distribution by Prediction Class

In [ ]:
COLORS = {"TP": "#2ecc71", "FP": "#e74c3c", "FN": "#f39c12", "TN": "#95a5a6"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score histogram per class
for pc in ("TP", "FP", "FN", "TN"):
    subset = scored[scored["pred_class"] == pc]
    if len(subset) == 0: continue
    axes[0].hist(subset["predicted_score"], bins=30, alpha=0.6, label=f"{pc} ({len(subset)})", color=COLORS[pc])
axes[0].set_xlabel("Predicted Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Score Distribution by Prediction Class")
axes[0].legend()
axes[0].axvline(x=0.5, color="black", linestyle="--", alpha=0.5, label="threshold")

# Score histogram: positives vs negatives
for label, name, color in [(1, "Positive (farm)", "#2ecc71"), (0, "Negative", "#95a5a6")]:
    subset = scored[scored["true_label"] == label]
    axes[1].hist(subset["predicted_score"], bins=30, alpha=0.6, label=f"{name} ({len(subset)})", color=color)
axes[1].set_xlabel("Predicted Score")
axes[1].set_ylabel("Count")
axes[1].set_title("Score Distribution by True Label")
axes[1].legend()
axes[1].axvline(x=0.5, color="black", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 2. Per-Split Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, split in enumerate(("train", "val", "test")):
    subset = scored[scored["split"] == split]
    if len(subset) == 0: continue
    cm = confusion_matrix(subset["true_label"], subset["predicted_label"], labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Not Farm", "Farm"])
    disp.plot(ax=axes[i], cmap="Blues", colorbar=False)
    axes[i].set_title(f"{split} (n={len(subset)})")

plt.tight_layout()
plt.show()

# Detailed report for test set
test = scored[scored["split"] == "test"]
if len(test) > 0:
    print("=== TEST SET ===")
    print(classification_report(test["true_label"], test["predicted_label"], target_names=["Not Farm", "Farm"]))

## 3. Visualize Sample Patches — TP / FP / FN / TN

Display RGB (B4, B3, B2) and NDVI for random samples from each prediction class.

In [ ]:
# Band order (channel-first): B2, B3, B4, B8, B11, B12, NDVI, NDBI, NDWI
# Shape: (9, 128, 128) — channels first, values ~0-4000 (raw reflectance)
BAND_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDBI", "NDWI"]
RGB_CH = [2, 1, 0]  # B4, B3, B2
NDVI_CH = 6

def load_patch(row):
    """Load .npy patch file for a scored candidate."""
    pp = row.get("patch_path", "")
    if not pp or pd.isna(pp):
        return None
    p = PATCHES_DIR / pp
    if not p.exists():
        return None
    return np.load(p)

def show_patch_rgb(ax, patch, title="", score=None):
    """Display RGB composite. Patch shape: (C, H, W)."""
    rgb = np.stack([patch[ch] for ch in RGB_CH], axis=-1)  # (H, W, 3)
    # Percentile stretch for better contrast
    p2, p98 = np.percentile(rgb, [2, 98])
    rgb = np.clip((rgb - p2) / max(p98 - p2, 1), 0, 1)
    ax.imshow(rgb)
    label = title
    if score is not None:
        label += f"\nscore={score:.3f}"
    ax.set_title(label, fontsize=9)
    ax.axis("off")

def show_patch_ndvi(ax, patch, title=""):
    """Display NDVI channel. Values are raw (not -1..1), use percentile stretch."""
    ndvi = patch[NDVI_CH]
    p2, p98 = np.percentile(ndvi, [2, 98])
    ax.imshow(ndvi, cmap="RdYlGn", vmin=p2, vmax=p98)
    ax.set_title(f"{title} NDVI", fontsize=9)
    ax.axis("off")

# Show a few test samples
test_samples = scored[scored["patch_path"].notna()].sample(4, random_state=42)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, (_, row) in enumerate(test_samples.iterrows()):
    patch = load_patch(row)
    if patch is None: continue
    show_patch_rgb(axes[0, i], patch, f"{row.get('pred_class','')} | {row.get('candidate_id','')[:20]}", row.get("predicted_score"))
    show_patch_ndvi(axes[1, i], patch, row.get("pred_class", ""))
axes[0, 0].set_ylabel("RGB", fontsize=12)
axes[1, 0].set_ylabel("NDVI", fontsize=12)
plt.tight_layout()
plt.show()

## 4. False Positive Deep Dive

What do FPs look like? Are they clustered in certain states? What are their scores?

In [ ]:
fps = scored[scored["pred_class"] == "FP"].copy()
print(f"Total FPs: {len(fps)}")
print(f"\nFP by split:\n{fps['split'].value_counts()}")

# FP score distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(fps["predicted_score"], bins=20, color="#e74c3c", alpha=0.7, edgecolor="white")
axes[0].set_title(f"FP Score Distribution (n={len(fps)})")
axes[0].set_xlabel("Predicted Score")
axes[0].axvline(x=0.5, color="black", linestyle="--")

# FP by state
if "state" in fps.columns:
    state_counts = fps.groupby("state").size().sort_values(ascending=False).head(15)
elif "candidate_id" in fps.columns:
    # Extract state from candidate_id pattern
    fps["_state"] = fps["candidate_id"].str.extract(r"_([A-Z]{2})_")
    state_counts = fps.groupby("_state").size().sort_values(ascending=False).head(15)
else:
    state_counts = pd.Series(dtype=int)

if len(state_counts) > 0:
    state_counts.plot.barh(ax=axes[1], color="#e74c3c", alpha=0.7)
    axes[1].set_title("FP by State (top 15)")
    axes[1].set_xlabel("Count")

# FP by confidence tier
if "confidence_tier" in fps.columns:
    tier_counts = fps["confidence_tier"].value_counts()
    tier_counts.plot.bar(ax=axes[2], color="#e74c3c", alpha=0.7)
    axes[2].set_title("FP by Confidence Tier")

plt.tight_layout()
plt.show()

## 5. High-Confidence FPs — Most Interesting Errors

These are the ones the model is most confident about but are wrong. Look at the patches to understand what's confusing the model.

In [ ]:
# Top 10 highest-confidence false positives
top_fps = fps.nlargest(10, "predicted_score")

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for i, (_, row) in enumerate(top_fps.iterrows()):
    patch = load_patch(row)
    if patch is None:
        axes[i].text(0.5, 0.5, "no patch", ha="center", va="center")
        axes[i].axis("off")
        continue
    show_patch_rgb(axes[i], patch,
        f"FP #{i+1}: {row.get('candidate_id', '')[:25]}",
        row.get("predicted_score"))

for i in range(len(top_fps), 10):
    axes[i].axis("off")

plt.suptitle("Top 10 Highest-Confidence False Positives", fontsize=14)
plt.tight_layout()
plt.show()

# Print details
print("\nTop FP details:")
cols = ["candidate_id", "predicted_score", "confidence_tier", "lat", "lng", "source"]
cols = [c for c in cols if c in top_fps.columns]
print(top_fps[cols].to_string(index=False))

## 5b. Verify FPs on Google Maps

Click the links to check if these "false positives" are actually farms the database missed.

In [ ]:
from IPython.display import display, HTML

top_fps = fps.nlargest(20, "predicted_score")

rows_html = ""
for _, row in top_fps.iterrows():
    lat, lng = row.get("lat", 0), row.get("lng", 0)
    score = row.get("predicted_score", 0)
    cid = row.get("candidate_id", "")
    gmaps = f"https://www.google.com/maps/@{lat},{lng},500m/data=!3m1!1e3"
    rows_html += (
        f"<tr>"
        f"<td>{cid[:30]}</td>"
        f"<td>{score:.3f}</td>"
        f"<td>{lat:.4f}, {lng:.4f}</td>"
        f'<td><a href="{gmaps}" target="_blank">Open in Google Maps</a></td>'
        f"</tr>"
    )

display(HTML(
    "<table border='1' style='border-collapse:collapse;'>"
    "<tr><th>Candidate ID</th><th>Score</th><th>Lat, Lng</th><th>Verify</th></tr>"
    f"{rows_html}</table>"
    "<br><i>If most of these are actually farms, the model is better than the metrics suggest — "
    "the 'false positives' are label noise from incomplete ground truth.</i>"
))

## 6. Band Statistics — FP vs TP

Compare mean band values to see if FPs have different spectral signatures.

In [ ]:
def compute_band_stats(subset, n_max=200):
    """Compute mean band values for a subset of candidates. Patch shape: (C, H, W)."""
    samples = subset.sample(min(n_max, len(subset)), random_state=42)
    band_means = []
    for _, row in samples.iterrows():
        patch = load_patch(row)
        if patch is None: continue
        band_means.append(patch.mean(axis=(1, 2)))  # mean over H, W per channel
    if not band_means:
        return None
    return np.array(band_means)

stats = {}
for pc in ("TP", "FP", "FN", "TN"):
    subset = scored[scored["pred_class"] == pc]
    if len(subset) == 0: continue
    s = compute_band_stats(subset)
    if s is not None:
        stats[pc] = s
        print(f"{pc}: computed stats for {len(s)} patches")

if stats:
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(BAND_NAMES))
    width = 0.2
    for i, (pc, s) in enumerate(stats.items()):
        means = s.mean(axis=0)
        stds = s.std(axis=0)
        ax.bar(x + i * width, means, width, label=pc, color=COLORS[pc], alpha=0.8, yerr=stds, capsize=2)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(BAND_NAMES)
    ax.set_ylabel("Mean Value")
    ax.set_title("Mean Band Values by Prediction Class")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 7. Side-by-Side: TP vs FP Comparison

Show pairs of TPs and FPs with similar scores to see what distinguishes them visually.

In [ ]:
N_PAIRS = 6
tps = scored[scored["pred_class"] == "TP"].nlargest(N_PAIRS, "predicted_score")
fps_top = fps.nlargest(N_PAIRS, "predicted_score")

fig, axes = plt.subplots(2, N_PAIRS, figsize=(N_PAIRS * 4, 8))

for i, (_, row) in enumerate(tps.iterrows()):
    if i >= N_PAIRS: break
    patch = load_patch(row)
    if patch is not None:
        show_patch_rgb(axes[0, i], patch, f"TP: {row.get('candidate_id', '')[:20]}", row.get("predicted_score"))
    else:
        axes[0, i].text(0.5, 0.5, "no patch", ha="center", va="center")
        axes[0, i].axis("off")

for i, (_, row) in enumerate(fps_top.iterrows()):
    if i >= N_PAIRS: break
    patch = load_patch(row)
    if patch is not None:
        show_patch_rgb(axes[1, i], patch, f"FP: {row.get('candidate_id', '')[:20]}", row.get("predicted_score"))
    else:
        axes[1, i].text(0.5, 0.5, "no patch", ha="center", va="center")
        axes[1, i].axis("off")

axes[0, 0].set_ylabel("True Positives", fontsize=12, rotation=0, labelpad=80)
axes[1, 0].set_ylabel("False Positives", fontsize=12, rotation=0, labelpad=80)
plt.suptitle("Highest-Scoring TPs vs FPs — What Does the Model See?", fontsize=14)
plt.tight_layout()
plt.show()